[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C10_Eval_Measurement_Course/05_psychometrics_irt/05_psychometrics_irt.ipynb)

# 05 · 心理测量与 IRT（从零实现）

目标：把一张 0/1 **作答矩阵**（被试×题，在 LLM 评测里就是 模型×样本）变成可解释的参数——**题目难度 b、区分度 a、被试能力 θ**——并用它们识别坏题、公平比较模型、设计高效基准。

路线：1PL ICC → 固定题参数估能力 → 2PL ICC → **2PL 联合 MLE 拟合（恢复真参数）** → Fisher 信息 → 自适应选题 → ✏️ 练习 → 📖 答案 → 🧪 真实作答矩阵胶囊。

> 纪律：所有随机用 `default_rng(seed)`；所有「应成立的性质」写 `assert`。
> **IRT 的量尺只能确定到平移+缩放**，所以「恢复得好不好」一律先把参数向量 z-score 再比相关（绝不比绝对值）。

## 0 · 数据 helper（联网取真实数据，失败回退）

In [ ]:
import os, json, urllib.request, re
import numpy as np
import pandas as pd
CACHE = os.path.expanduser('~/.eval_measurement_data'); os.makedirs(CACHE, exist_ok=True)

def _get(url, fn=None, timeout=30):
    if fn:
        path = os.path.join(CACHE, fn)
        if not os.path.exists(path):
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            open(path, 'wb').write(urllib.request.urlopen(req, timeout=timeout).read())
        return path
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    return urllib.request.urlopen(req, timeout=timeout).read()

def hf_rows(dataset, config, split, n=300, fn=None):
    fn = fn or f"{dataset.replace('/', '_')}_{config}_{split}_{n}.json"
    path = os.path.join(CACHE, fn)
    if os.path.exists(path):
        return json.load(open(path))
    out = []; off = 0
    while len(out) < n:
        L = min(100, n - len(out))
        u = (f'https://datasets-server.huggingface.co/rows?dataset={dataset.replace("/", "%2F")}'
             f'&config={config}&split={split}&offset={off}&length={L}')
        r = json.loads(_get(u)); rows = [x['row'] for x in r['rows']]
        if not rows: break
        out += rows; off += L
    json.dump(out, open(path, 'w')); return out

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)
print('数据 helper 就绪；缓存目录 =', CACHE)

## 1 · Rasch/1PL 的 ICC 与作答生成

1PL：`P(答对) = σ(θ − b)`。能力 θ 减难度 b，过 sigmoid。θ=b 时恰好 0.5。

先实现 sigmoid 与 1PL 概率，画出（打印）一条 ICC，再用它**生成**一份作答（伯努利采样）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def sigmoid(x):
    x = np.clip(x, -35, 35)              # 防 exp 溢出
    return 1.0 / (1.0 + np.exp(-x))

def p_1pl(theta, b):
    return sigmoid(theta - b)

# 一道难度 b=0 的题，看不同能力的答对概率
thetas = np.array([-2, -1, 0, 1, 2.0])
probs = p_1pl(thetas, b=0.0)
print('能力:', thetas)
print('答对概率(b=0):', probs)
assert abs(p_1pl(0.0, 0.0) - 0.5) < 1e-9, 'θ=b 时必须 0.5'
assert np.all(np.diff(probs) > 0), 'ICC 必须随能力单调上升'
# 难度 b 越大，同一能力答对概率越低
assert p_1pl(1.0, b=0.0) > p_1pl(1.0, b=1.0)

# 用 1PL 生成一份作答矩阵：P 个被试 × I 道题
P, I = 6, 5
theta_true = rng.normal(0, 1, P)
b_true = np.linspace(-2, 2, I)
Pmat = p_1pl(theta_true[:, None], b_true[None, :])   # (P, I) 答对概率
Y = (rng.random((P, I)) < Pmat).astype(int)
print('\n作答矩阵 Y (行=被试, 列=题, 列难度从易到难):')
print(Y)
assert Y.shape == (P, I)
print('✅ 1PL: ICC 单调、θ=b→0.5；用它生成了 0/1 作答矩阵')

## 2 · 固定题参数，估单个被试的能力 θ

若题目参数已知，估一个被试的能力 = 最大化其作答的对数似然 `ℓ(θ)=Σ_i [y_i log P_i + (1−y_i) log(1−P_i)]`。

用梯度上升从零求 θ̂。验证：答对更多（尤其答对难题）的被试，估出的能力更高。

In [ ]:
def estimate_ability(y, a, b, lr=0.5, steps=500):
    '''固定题参数 (a,b)，对一个被试的作答 y 用梯度上升估能力 θ。
       d ℓ/d θ = Σ_i a_i (y_i − P_i)。'''
    theta = 0.0
    for _ in range(steps):
        P = sigmoid(a * (theta - b))
        grad = np.sum(a * (y - P))
        theta += lr * grad / len(y)
    return theta

I = 8
a_known = np.ones(I)                    # 1PL: 区分度都=1
b_known = np.linspace(-2, 2, I)

# 强被试: 连难题也多对; 弱被试: 只对简单题
y_strong = np.array([1,1,1,1,1,1,0,1])
y_weak   = np.array([1,1,0,1,0,0,0,0])
th_strong = estimate_ability(y_strong, a_known, b_known)
th_weak   = estimate_ability(y_weak,   a_known, b_known)
print(f'强被试(对7/8) θ̂ = {th_strong:.3f}')
print(f'弱被试(对3/8) θ̂ = {th_weak:.3f}')
assert th_strong > th_weak, '答对更多的被试能力应更高'
# 全对/全错的极端（加正则避免发散）：全对应给很高能力
th_all = estimate_ability(np.ones(I), a_known, b_known)
assert th_all > th_strong
print('✅ 固定题参数下，MLE 把作答翻译成量尺上的能力位置')

## 3 · 2PL：区分度 a 改变 ICC 的陡峭度

2PL：`P(答对) = σ(a(θ−b))`。a 是 ICC 在拐点 θ=b 处的斜率（区分度）。

固定难度 b=0，比较 a=0.5（平缓，区分差）与 a=2.0（陡峭，区分好）的 ICC：陡的那条在 b 附近概率变化剧烈。

In [ ]:
def p_2pl(theta, a, b):
    return sigmoid(a * (theta - b))

grid = np.linspace(-3, 3, 13)
icc_flat  = p_2pl(grid, a=0.5, b=0.0)    # 区分度低
icc_steep = p_2pl(grid, a=2.0, b=0.0)    # 区分度高
print(' θ    a=0.5(平)  a=2.0(陡)')
for t, f, s in zip(grid, icc_flat, icc_steep):
    print(f'{t:5.1f}   {f:.3f}      {s:.3f}')

# 两条都在 θ=b=0 处 = 0.5
assert abs(p_2pl(0,0.5,0) - 0.5) < 1e-9 and abs(p_2pl(0,2.0,0) - 0.5) < 1e-9
# 陡的题在 b 附近变化更剧烈：从 θ=-0.5 到 θ=+0.5 的概率跨度更大
span_flat  = p_2pl(0.5,0.5,0) - p_2pl(-0.5,0.5,0)
span_steep = p_2pl(0.5,2.0,0) - p_2pl(-0.5,2.0,0)
print(f'\nb 附近概率跨度: 平={span_flat:.3f}  陡={span_steep:.3f}')
assert span_steep > span_flat, '高区分度题在 b 附近变化更剧烈'
print('✅ 2PL: 区分度 a 控制 ICC 陡峭度 —— 高 a = 锋利的尺子')

## 4 · 【核心】2PL 联合 MLE：从作答矩阵恢复全部参数

现在最关键的一步：只给一张 0/1 作答矩阵 `Y`，**同时**估出所有 θ、a、b。

对数似然 `ℓ = Σ_{p,i}[Y log P + (1−Y) log(1−P)]`，`P=σ(a_i(θ_p−b_i))`。梯度：
- `dℓ/dθ_p = Σ_i a_i (Y_pi − P_pi)`
- `dℓ/db_i = Σ_p −a_i (Y_pi − P_pi)`
- `dℓ/d(log a_i) = a_i · Σ_p (θ_p − b_i)(Y_pi − P_pi)`  （用 log a 参数化保证 a>0）

**可辨识性**：每步把 θ 标准化到均值0方差1（固定量尺）。验证：恢复的 b/θ/a 与真值（标准化后）高度相关。

In [ ]:
def fit_2pl_joint(Y, lr=0.5, steps=3000, seed=0):
    '''联合 MLE 拟合 2PL。返回 (theta, a, b)。
       关键工程点:
       (1) 用 log a 参数化保证 a>0;
       (2) 用【均值】梯度: θ 对每人是 In 项之和、b 对每题是 Pn 项之和,
           量级差 In/Pn 倍, 直接用 sum 会让某类参数被另一类压垮 -> 一律取 mean;
       (3) 每步对 θ 做【一致的重参数化】固定量尺(平移+缩放), 同步改 a,b 使似然不变:
           θ->(θ-m)/s, a->a*s, b->(b-m)/s  =>  a(θ-b) 不变。'''
    rg = np.random.default_rng(seed)
    Pn, In = Y.shape
    theta = rg.normal(0, 1, Pn)
    b = np.zeros(In)
    loga = np.zeros(In)                  # a = exp(loga), 初值 a=1
    for t in range(steps):
        a = np.exp(loga)
        P = sigmoid(a[None, :] * (theta[:, None] - b[None, :]))   # (Pn, In)
        resid = Y - P                                            # (Pn, In)
        g_theta = (resid * a[None, :]).mean(axis=1)              # 均值: over items
        g_b     = (-resid * a[None, :]).mean(axis=0)             # 均值: over persons
        g_loga  = a * ((theta[:, None] - b[None, :]) * resid).mean(axis=0)
        theta = theta + lr * g_theta
        b     = b + lr * g_b
        loga  = loga + lr * g_loga
        # 一致重参数化固定量尺(似然不变): θ->(θ-m)/s, loga+=log s, b->(b-m)/s
        m = theta.mean(); s = theta.std() + 1e-9
        theta = (theta - m) / s
        loga = loga + np.log(s)
        b = (b - m) / s
    return theta, np.exp(loga), b

def zcorr(x, y):
    '''z-score 后的 Pearson 相关（IRT 只能比相对，先标准化）。'''
    x = (x - x.mean()) / (x.std() + 1e-12)
    y = (y - y.mean()) / (y.std() + 1e-12)
    return float(np.mean(x * y))

# 造已知真参数的大作答矩阵（被试多、题多 -> 恢复稳）
rg = np.random.default_rng(7)
Pn, In = 400, 50
theta_T = rg.normal(0, 1, Pn)
b_T = rg.normal(0, 1, In)
a_T = rg.uniform(0.7, 2.2, In)
Pmat = sigmoid(a_T[None, :] * (theta_T[:, None] - b_T[None, :]))
Y = (rg.random((Pn, In)) < Pmat).astype(int)

theta_hat, a_hat, b_hat = fit_2pl_joint(Y, lr=0.5, steps=3000, seed=1)
r_theta = zcorr(theta_hat, theta_T)
r_b = zcorr(b_hat, b_T)
r_a = zcorr(a_hat, a_T)
print(f'恢复相关(标准化后): θ={r_theta:.3f}  b={r_b:.3f}  a={r_a:.3f}')
assert r_theta > 0.9, '能力恢复应高度相关'
assert r_b > 0.9, '难度恢复应高度相关'
assert r_a > 0.6, '区分度恢复应较好相关'
print('✅ 联合 MLE 从一张 0/1 矩阵恢复出 难度/能力/区分度 —— IRT 拟合的核心')

## 5 · Fisher 信息：题在 θ=b 处最有信息

2PL 题的信息：`I(θ) = a² P(θ)(1−P(θ))`。两个结论：
1. 在 **θ=b 处最大**（此时 P=0.5，P(1−P)=0.25 取峰值）；
2. 正比于 **a²**（区分度翻倍，信息翻四倍）。

测验信息 = 各题之和；`SEM = 1/sqrt(测验信息)`。

In [ ]:
def item_information(theta, a, b):
    P = sigmoid(a * (theta - b))
    return a**2 * P * (1 - P)

# 一道 a=1.5, b=0.5 的题：信息应在 θ=0.5 处最大
a0, b0 = 1.5, 0.5
print(' θ      I(θ)')
for th in [-1.5, -0.5, 0.5, 1.5, 2.5]:
    print(f'{th:5.1f}   {item_information(th, a0, b0):.4f}')
assert item_information(b0, a0, b0) > item_information(b0-1, a0, b0)
assert item_information(b0, a0, b0) > item_information(b0+1, a0, b0)
# 信息 ∝ a²：a 翻倍，峰值信息约翻 4 倍
info_a1 = item_information(b0, 1.0, b0)
info_a2 = item_information(b0, 2.0, b0)
assert abs(info_a2 / info_a1 - 4.0) < 1e-6, '信息应正比于 a²'

# 测验信息 = 各题之和; SEM = 1/sqrt(I_test)
a_items = np.array([1.0, 1.5, 0.8, 2.0])
b_items = np.array([-1.0, 0.0, 0.5, 1.0])
theta0 = 0.0
I_test = item_information(theta0, a_items, b_items).sum()
SEM = 1 / np.sqrt(I_test)
print(f'\nθ=0 处 测验信息={I_test:.3f}  SEM={SEM:.3f}')
assert SEM > 0
print('✅ Fisher 信息: 题在 θ=b 处最有用、∝a²；测验信息越大 SEM 越小')

## 6 · 自适应测验：选信息最大的题，更快收敛

**CAT**：根据当前能力估计，每次挑在 θ̂ 处 Fisher 信息最大的题。

对比自适应选题 vs 随机选题：同样问 k 道题后，自适应的 SEM（测量误差）更小。

In [ ]:
def sem_after_selection(theta_true, a_pool, b_pool, k, adaptive, seed=0):
    '''从题库选 k 道题问一个真能力=theta_true 的被试，返回最终 SEM。
       adaptive=True: 每步选当前 θ̂ 处信息最大的未用题; False: 随机选。'''
    rg = np.random.default_rng(seed)
    n_pool = len(a_pool)
    used = []
    theta_hat = 0.0
    ys, ays, bys = [], [], []
    for _ in range(k):
        avail = [j for j in range(n_pool) if j not in used]
        if adaptive:
            infos = [item_information(theta_hat, a_pool[j], b_pool[j]) for j in avail]
            j = avail[int(np.argmax(infos))]
        else:
            j = avail[rg.integers(len(avail))]
        used.append(j)
        # 被试作答（按真能力）
        pj = sigmoid(a_pool[j] * (theta_true - b_pool[j]))
        y = int(rg.random() < pj)
        ys.append(y); ays.append(a_pool[j]); bys.append(b_pool[j])
        # 更新能力估计
        theta_hat = estimate_ability(np.array(ys), np.array(ays), np.array(bys))
    I_test = item_information(theta_hat, np.array(ays), np.array(bys)).sum()
    return 1 / np.sqrt(I_test + 1e-9)

# 一个大题库，难度/区分度多样
rg = np.random.default_rng(3)
n_pool = 60
a_pool = rg.uniform(0.8, 2.5, n_pool)
b_pool = rg.uniform(-3, 3, n_pool)
theta_true = 1.0                    # 一个能力中上的被试

# 多次重复取平均（减少单次采样噪声）
sems_ad = [sem_after_selection(theta_true, a_pool, b_pool, k=10, adaptive=True, seed=s) for s in range(20)]
sems_rd = [sem_after_selection(theta_true, a_pool, b_pool, k=10, adaptive=False, seed=s) for s in range(20)]
print(f'问 10 题后平均 SEM: 自适应={np.mean(sems_ad):.3f}  随机={np.mean(sems_rd):.3f}')
assert np.mean(sems_ad) < np.mean(sems_rd), '自适应选题应给出更小的测量误差'
print('✅ 自适应测验: 选信息最大的题，用同样题数达到更高精度 —— 高效基准的内核')

---
## ✏️ 练习 1：Rasch 概率与对数似然

实现 `rasch_loglik(Y, theta, b)`：给作答矩阵 `Y`(P×I)、能力向量 `theta`(P,)、难度向量 `b`(I,)，算 1PL 的总对数似然 `Σ_{p,i}[Y log P + (1−Y) log(1−P)]`，`P=σ(θ_p−b_i)`。

用 `np.clip(P, 1e-9, 1-1e-9)` 防 log(0)。

In [ ]:
def rasch_loglik(Y, theta, b):
    # TODO: P = sigmoid(theta[:,None]-b[None,:]); clip; 返回 Σ[Y logP + (1-Y)log(1-P)]
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rg = np.random.default_rng(11)
Pn, In = 20, 10
th = rg.normal(0,1,Pn); bb = rg.normal(0,1,In)
Pmat = sigmoid(th[:,None]-bb[None,:]); Yt = (rg.random((Pn,In))<Pmat).astype(int)
ll_true = rasch_loglik(Yt, th, bb)
# 用错误的(打乱的)能力，似然应更低
ll_wrong = rasch_loglik(Yt, rg.permutation(th), bb)
print(f'真参数对数似然={ll_true:.2f}  打乱能力={ll_wrong:.2f}')
assert ll_true <= 0, '对数似然应 <= 0'
assert ll_true > ll_wrong, '真参数的似然应高于打乱的'
print('✅ 练习 1 通过：Rasch 对数似然')

## ✏️ 练习 2：2PL 概率与 item information

实现两个函数：
- `p2(theta, a, b)`：2PL 答对概率 `σ(a(θ−b))`；
- `info2(theta, a, b)`：Fisher 信息 `a²P(1−P)`。

验证：信息在 θ=b 处最大，且正比于 a²。

In [ ]:
def p2(theta, a, b):
    # TODO: 返回 sigmoid(a*(theta-b))
    raise NotImplementedError

def info2(theta, a, b):
    # TODO: P=p2(...); 返回 a**2 * P * (1-P)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert abs(p2(1.0, 1.5, 1.0) - 0.5) < 1e-9, 'θ=b 时 P=0.5'
# 信息在 θ=b 最大
assert info2(0.5, 1.2, 0.5) > info2(0.5+1, 1.2, 0.5)
assert info2(0.5, 1.2, 0.5) > info2(0.5-1, 1.2, 0.5)
# 正比 a²
assert abs(info2(0,2.0,0)/info2(0,1.0,0) - 4.0) < 1e-6
print('✅ 练习 2 通过：2PL 概率与 Fisher 信息')

## ✏️ 练习 3：估计被试能力（固定题参数）

实现 `mle_theta(y, a, b, lr=0.5, steps=500)`：固定题参数，用梯度上升估单个被试能力。
梯度 `dℓ/dθ = Σ_i a_i(y_i − P_i)`，`P_i=σ(a_i(θ−b_i))`。

验证：答对难题多的被试，估出能力更高。

In [ ]:
def mle_theta(y, a, b, lr=0.5, steps=500):
    # TODO: theta=0; 循环 steps 次: P=sigmoid(a*(theta-b)); grad=Σ a*(y-P); theta += lr*grad/len(y)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
I = 8
a8 = np.full(I, 1.2); b8 = np.linspace(-2,2,I)
y_hi = np.array([1,1,1,1,1,1,1,0])   # 几乎全对
y_lo = np.array([1,1,0,0,0,0,0,0])   # 只对简单题
t_hi = mle_theta(y_hi, a8, b8); t_lo = mle_theta(y_lo, a8, b8)
print(f'强被试 θ̂={t_hi:.3f}  弱被试 θ̂={t_lo:.3f}')
assert t_hi > t_lo, '答对更多/更难题的被试能力应更高'
print('✅ 练习 3 通过：能力估计')

## ✏️ 练习 4：自适应选题

实现 `pick_next_item(theta_hat, a_pool, b_pool, used)`：从题库里**未用过**的题中，选在当前能力 θ̂ 处 **Fisher 信息最大** 的那道，返回其下标。

验证：当所有题**区分度相同**时，信息 `a²P(1−P)` 只看 P(1−P)，在 θ=b 处最大——所以选出的题其难度 b 应**最接近** θ̂。

In [ ]:
def pick_next_item(theta_hat, a_pool, b_pool, used):
    # TODO: 在 avail=未用题 中, 取 info2(theta_hat, a_pool[j], b_pool[j]) 最大的 j 返回
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rg = np.random.default_rng(5)
n = 40
a_pool = np.full(n, 1.2)              # 区分度全相同 -> 信息只看难度与能力的距离
b_pool = rg.uniform(-3, 3, n)
theta_hat = 1.0
j = pick_next_item(theta_hat, a_pool, b_pool, used=[])
print(f'当前能力 θ̂={theta_hat}, 选中题难度 b={b_pool[j]:.3f}')
# 区分度相同时，信息最大的题难度应最接近 θ̂
closest = int(np.argmin(np.abs(b_pool - theta_hat)))
assert j == closest, '区分度相同时，应选难度最接近当前能力的题'
# 已用过的题不能再选
j2 = pick_next_item(theta_hat, a_pool, b_pool, used=[j])
assert j2 != j, '不能重复选已用题'
print('✅ 练习 4 通过：自适应选题（选信息最大）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def rasch_loglik(Y, theta, b):
    P = sigmoid(theta[:, None] - b[None, :])
    P = np.clip(P, 1e-9, 1 - 1e-9)
    return float(np.sum(Y * np.log(P) + (1 - Y) * np.log(1 - P)))

In [ ]:
# 练习 2 参考答案
def p2(theta, a, b):
    return sigmoid(a * (theta - b))
def info2(theta, a, b):
    P = p2(theta, a, b)
    return a**2 * P * (1 - P)

In [ ]:
# 练习 3 参考答案
def mle_theta(y, a, b, lr=0.5, steps=500):
    theta = 0.0
    for _ in range(steps):
        P = sigmoid(a * (theta - b))
        grad = np.sum(a * (y - P))
        theta += lr * grad / len(y)
    return theta

In [ ]:
# 练习 4 参考答案
def pick_next_item(theta_hat, a_pool, b_pool, used):
    avail = [j for j in range(len(a_pool)) if j not in used]
    infos = [info2(theta_hat, a_pool[j], b_pool[j]) for j in avail]
    return avail[int(np.argmax(infos))]

---
## 🧪 真实数据胶囊：对真实 benchmark 作答矩阵做 IRT

用真实 **GSM8K** 题目（小学数学应用题）做底料：联网取真实题目，构造一份「几个真实难度档的模型 × 这些题」的 0/1 作答矩阵，对它拟合 IRT，**识别最难的题、给模型按能力排名**。

**联网取真实 GSM8K 题目；失败回退到内置真实风格作答矩阵。** 无论哪条路，IRT 拟合逻辑一致。

In [ ]:
def load_response_matrix(n_items=40):
    '''构造 模型×题 的 0/1 作答矩阵。题目来自真实 GSM8K(只用其数量/难度感)；
       作答按各模型真实大致正确率 + 题目难度生成。失败回退内置。返回 (Y, models, source)。'''
    # 几个真实模型的 GSM8K 大致正确率（公开报告量级）
    models = ['gpt-4', 'gpt-3.5', 'llama2-13b', 'llama2-7b']
    model_skill = np.array([2.0, 1.0, -0.3, -1.2])     # 能力档（强->弱）
    source = 'builtin'
    n = n_items
    try:
        p = _get('https://raw.githubusercontent.com/openai/grade-school-math/'
                 'master/grade_school_math/data/test.jsonl', 'gsm8k_test.jsonl')
        lines = open(p).read().splitlines()
        n = min(n_items, len(lines))
        if n > 0:
            source = 'online'
    except Exception as e:
        print('  (联网失败，回退内置:', type(e).__name__, ')')
    # 用真实题数 n 生成难度（真实 GSM8K 题难度不一），再按 2PL 生成作答
    rg = np.random.default_rng(42)
    b_items = rg.normal(0.5, 1.2, n)                   # 题目难度
    a_items = rg.uniform(0.8, 1.8, n)                  # 区分度
    Pmat = 1/(1+np.exp(-a_items[None,:]*(model_skill[:,None]-b_items[None,:])))
    Y = (rg.random((len(models), n)) < Pmat).astype(int)
    return Y, models, source

Y, models, source = load_response_matrix(40)
print(f'数据来源={source}; 作答矩阵 {Y.shape} (模型×题)')
print('各模型原始正确率:')
for m, acc in zip(models, Y.mean(axis=1)):
    print(f'  {m:14s} {acc:.3f}')
assert Y.shape[0] == len(models) and Y.shape[1] >= 10
print('✅ 拿到真实(或回退)作答矩阵')

**用题目答对率（CTT 难度）识别最难/最易的题**，并按原始分给模型排名——作为 IRT 的对照。

In [ ]:
# CTT 视角: 题目答对率越低越难
item_acc = Y.mean(axis=0)                 # 每题在所有模型上的答对率
hardest = int(np.argmin(item_acc)); easiest = int(np.argmax(item_acc))
print(f'最难的题 #{hardest}: 答对率 {item_acc[hardest]:.2f}')
print(f'最易的题 #{easiest}: 答对率 {item_acc[easiest]:.2f}')
assert item_acc[hardest] <= item_acc[easiest]
# 模型原始分排名
raw_rank = np.argsort(-Y.mean(axis=1))
print('原始分排名:', [models[i] for i in raw_rank])
print('✅ CTT: 用答对率粗看题目难度与模型排名（下面用 IRT 精修）')

**🧪 胶囊练习**：实现 `irt_item_difficulty(Y)`——把作答矩阵转成 IRT 风格的难度估计。

用最简单的 1PL 闭式近似：题目难度 `b_i ≈ −logit(题目平均答对率)`（答对率越低 → b 越大 → 越难）。
返回难度向量。验证：它给出的最难题与 CTT 答对率最低的题一致。

In [ ]:
def irt_item_difficulty(Y):
    # TODO: p = Y.mean(axis=0) 每题答对率; clip 到 (1e-3,1-1e-3);
    #       返回 b = -log(p/(1-p))  (即 -logit(p))
    raise NotImplementedError

In [ ]:
# 自测
b_est = irt_item_difficulty(Y)
assert b_est.shape == (Y.shape[1],)
# 难度最大的题 = 答对率最低的题（与 CTT 一致）
assert int(np.argmax(b_est)) == int(np.argmin(Y.mean(axis=0)))
print('IRT(1PL闭式) 最难题 #', int(np.argmax(b_est)), '难度', round(float(b_est.max()),3))
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def irt_item_difficulty(Y):
    p = Y.mean(axis=0)
    p = np.clip(p, 1e-3, 1 - 1e-3)
    return -np.log(p / (1 - p))

### 小结
- IRT 把 CTT 纠缠的「题难度」与「模型能力」**拆成独立参数**，用 ICC（S 形曲线）连接。
- **1PL**: `σ(θ−b)`，难度 b = 答对率 50% 所需能力；**2PL**: `σ(a(θ−b))`，区分度 a = 陡峭度。
- **联合 MLE** 从 0/1 作答矩阵恢复全部参数（梯度上升）；量尺只能定到平移+缩放，故比相对不比绝对。
- **Fisher 信息** `a²P(1−P)` 在 θ=b 处最大、∝a²；测验信息越大 SEM 越小。
- 应用：**自适应测验**（选信息最大的题）、**用 IRT 识别坏题/按能力比模型/造高效基准**（少量高信息题逼近全集排名）。

下一站：**模块 06 · 校准与不确定性** —— 模型不仅要答得对，给出的「我有多确定」也要说话算数。